# Lab 02 — A tiny agent loop (plan → act → observe)

Original OfferReady lab. Build the agent reasoning loop from scratch so the
LangGraph concepts (state, nodes, conditional edges, a step cap) are concrete.
No dependencies — the same shape maps directly onto LangGraph nodes/edges.

**You will:** define shared state, write plan/act/observe nodes, route with a
condition, and cap iterations so it can't loop forever.

## 1. Shared state

In LangGraph, a typed state object flows through nodes and each node returns
updates that are merged (reducers). Here it's a plain dict.

In [ ]:
def new_state(question):
    return {"question": question, "messages": [], "steps": 0, "answer": None}

## 2. A tool the agent can call

Narrow and read-only — the agent proposes a call; our code decides what runs.

In [ ]:
ORDERS = {"A1": {"status": "shipped", "eta": "tomorrow"},
          "A2": {"status": "processing", "eta": "3 days"}}

def get_order(order_id: str) -> dict:
    "READ-ONLY tool: look up one order."
    return ORDERS.get(order_id, {"status": "unknown"})

## 3. The nodes

`plan` decides the next action, `act` runs the tool, `observe` records the result
and decides whether we're done.

In [ ]:
import re

def plan(state):
    # Trivial planner: if the question names an order id, plan to look it up.
    m = re.search(r"\b(A\d)\b", state["question"])
    action = {"tool": "get_order", "arg": m.group(1)} if m else {"tool": None}
    state["messages"].append(("plan", action))
    state["_action"] = action
    return state

def act(state):
    action = state["_action"]
    if action["tool"] == "get_order":
        result = get_order(action["arg"])
        state["messages"].append(("tool", result))
        state["_result"] = result
    state["steps"] += 1
    return state

def observe(state):
    result = state.get("_result")
    if result:
        state["answer"] = f"Order is {result['status']} (eta {result.get('eta','n/a')})."
    else:
        state["answer"] = "I couldn't find an order id in your question."
    return state

## 4. The loop with a hard step cap

The routing decides whether to loop or finish. **Always cap iterations** so an
agent can't spin forever or run up cost.

In [ ]:
MAX_STEPS = 4

def run(question):
    state = new_state(question)
    while True:
        state = plan(state)
        state = act(state)
        state = observe(state)
        # done when we have an answer, or hit the cap (guardrail)
        if state["answer"] is not None or state["steps"] >= MAX_STEPS:
            return state

final = run("What's the status of order A1?")
print("answer:", final["answer"])
print("steps:", final["steps"])
final["messages"]

## Mapping to LangGraph

- `new_state` → a `TypedDict` state with reducers.
- `plan`/`act`/`observe` → graph **nodes**.
- the done-or-cap check → a **conditional edge** (`route` returning `"plan"` or `END`).
- `get_order` → a bound **tool** (keep it least-privilege / read-only).
- add **checkpointing** to pause before a write tool for human approval.

See the Study Guide, Chapter 8 (LangGraph).